# Setup

In [ ]:
# Install pychometrics directly from GitHub
!pip install git+https://github.com/childmindresearch/llm_tracker

## ⚠️ Important: Data Privacy and Anonymization

Before using this tutorial with your own data, you should ensure that any sensitive or personally identifiable information (PII) has been removed or anonymized.

Because this package sends text to an LLM API for analysis, **any raw text you provide may leave your local environment** depending on your configuration. For this reason, you should **not upload identifiable or sensitive data unless it has been properly anonymized first**.

### Recommended approaches

If you are working with real data, we strongly recommend anonymizing it locally before running the analysis. We recommend: 

- https://github.com/microsoft/presidio
- https://github.com/childmindresearch/anonymize-pii

These tools can help remove names, locations, emails, and other identifying information before any LLM processing occurs.

### If you prefer not to use real data

If you are not comfortable anonymizing your own dataset, you can use the **publicly available Reddit data provided in this repository**, which is safe to use for demonstration purposes.

---

**In short:**  
✔ Use anonymized data whenever possible  
✔ When in doubt, use the provided sample dataset

# LLM Tracker Tutorial

**LLM Trackers** is a Python package for identifying instances of psychological constructs in qualitative text data — such as clinical documents, therapy transcripts, or open-ended survey responses — using large language models (LLMs).

The core workflow has four steps:

1. **LLM Coding** — run a language model over a directory of document files to identify and extract quotes that are instances of psychological constructs defined in a codebook
2. **Human Coding** — produce a second set of codings for comparison. In production this is real human-coded data; for testing purposes a second LLM run can serve as a stand-in
3. **Comparison** — align the two sets of codings instance-by-instance using an LLM matcher, classifying each instance as a true positive, false positive, or false negative
4. **Summary Tables** — compute per-document, concatenated, and weighted performance metrics across constructs

**Prerequisites:**
- An [OpenRouter](https://openrouter.ai) account and API key. OpenRouter provides access to a wide range of models (GPT-4o, Gemini, Claude, Llama, etc.) through a single API. An API key will be a unique code allowing you to access openrouter's models via code. It is unique to your account so save it somewhere safe. Create an account, add credits, and generate a key at https://openrouter.ai/settings/keys
- A directory of document files (`.txt` or `.csv`), one file per document
- A JSON codebook defining the psychological constructs to identify (see `suicide_risk_codebook.json` for an example)

## Setup

Import the core classes and functions used throughout this tutorial:
- `PychometricsAnalyzer` — runs LLM coding over a directory of document files
- `PychoMetricsComparer` — aligns and compares two sets of codings (human vs LLM)
- `format_comparison_table` — converts raw comparison output into a row-level DataFrame
- `compute_summary_tables` — computes per-document, concatenated, and weighted summary metrics
- `format_weighted_summary` — formats the weighted summary table for display
- `format_concatenated` — formats the concatenated table for display

In [ ]:
# first we import everything we'll need for the tutorial.
import pandas as pd
from llm_tracker import PychometricsAnalyzer
from llm_tracker.comparison import (
    PychometricsComparer,
    compute_summary_tables,
    format_comparison_table,
    format_concatenated,
    format_per_interview,
    format_weighted_summary,
)
from llm_tracker.file_handlers import load_dedoose_dataframe
from llm_tracker.utils import format_coding_table


/Users/freymon.perez/Projects/GitHub/pychometrics/src/pychometrics/models.py:41: UserWarning: Field name "construct" in "ConstructInstance" shadows an attribute in parent "BaseModel"
  class ConstructInstance(BaseModel):


## Paths and Configuration

Set all paths and credentials here before running any other cells. This is the only cell you need to edit for a new project.

- `input_dir` — directory containing your document files. Each file should be a single document in `.txt` or `.csv` format
- `codebook_path` — path to your JSON codebook. This defines the psychological constructs the LLM will look for and extract quotes for
- `api_key` — your OpenRouter API key. **Important:** OpenRouter API keys are case-sensitive — copy the key exactly as shown in your OpenRouter dashboard
- `llm_model` — the model to use for both coding and comparison. Any model listed at https://openrouter.ai/models can be passed here. You can use different models for coding vs comparison if desired (see Step 3)

In [ ]:

input_dir     = "/Users/freymon.perez/Projects/GitHub/pychometrics/sample_data/reddit_autism_anxiety_depression.csv"
codebook_path = "suicide_risk_codebook.json"


# API Key and Model selection.
# Uses OpenRouter. Any model available on OpenRouter can be passed to model_name.
# At the descretion of the user different models can be used for encoding or comparison, 
# In this tutorial we'll stick to one model for all tasks.
api_key   = "/Users/freymon.perez/Desktop/openrouter_api.env"
llm_model = "google/gemini-3-flash-preview"

## Step 1: LLM Coding

This step runs the LLM over every file in `input_dir` and extracts quotes that are instances of each construct defined in the codebook. Each document is processed independently, and results are saved to a timestamped output directory so runs are never overwritten.

The output directory has the following structure:
```
LLM_coding_YYYY-MM-DD_HHMMSS/
    codings/     <- one JSON per document, containing extracted construct instances
    metadata/    <- token usage, latency, and model info per document
    errors/      <- any documents that failed, with error messages
    README.md
```

Each coding JSON contains a list of instances, where each instance includes the construct name, the extracted quote, character-level indices into the source text, and a confidence score from the LLM.

**Note:** After running, copy the output directory path from the printout below and update `llm_encoding_dir` in Step 3.

In [ ]:
analyzer = PychometricsAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

#Set the data type here! Select either reddit if your document is a reddit dataframe or
#interview if you have a directory with single interview documents. 
data_type = "csv"

if data_type == "csv":
    results_llm, metadata_llm, errors_llm = analyzer.analyze_csv(
        csv_path="reddit_autism_anxiety_depression.csv",
        text_column="post",
        subreddit_column="subreddit",
        author_column="author",
        codebook_path=codebook_path,
        output_dir="LLM_reddit_coding",
    )
elif data_type =="interview":
    results_llm, metadata_llm, errors_llm = analyzer.analyze_directory(
        input_dir=input_dir,
        codebook_path=codebook_path,
        output_dir="LLM_coding",  # timestamped suffix added automatically
    )
else: 
    print("Please set data type to either csv or interview!")

Processing [1/60]: anxiety_An0nus3r123
  ✓ Found 4 construct instances
  Progress: Successful 1/60; Errors 0/60
Processing [2/60]: anxiety_Anon1234567899754
  ✓ Found 2 construct instances
  Progress: Successful 2/60; Errors 0/60
Processing [3/60]: anxiety_vwerfel
  ✓ Found 3 construct instances
  Progress: Successful 3/60; Errors 0/60
Processing [4/60]: anxiety_LostWildOne
  ✓ Found 8 construct instances
  Progress: Successful 4/60; Errors 0/60
Processing [5/60]: anxiety_frauleinfarts
  ✓ Found 2 construct instances
  Progress: Successful 5/60; Errors 0/60
Processing [6/60]: anxiety_neo-mancer
  ✓ Found 2 construct instances
  Progress: Successful 6/60; Errors 0/60
Processing [7/60]: anxiety_4ssCr4ck
  ✓ Found 8 construct instances
  Progress: Successful 7/60; Errors 0/60
Processing [8/60]: anxiety_NeptuneOW
  ✓ Found 3 construct instances
  Progress: Successful 8/60; Errors 0/60
Processing [9/60]: anxiety_zhvng666
  ✓ Found 7 construct instances
  Progress: Successful 9/60; Errors 0/

## The analyzer will always return and save three dictionaries. 
    * results - What the llm actually coded
    * metadata - Information on how the llm was prompted
    * errors - Information on any calls to the llm that failed for some reason.

These will be found as JSONs in the directory you specified above as the output_dir. 

In [4]:
format_coding_table(results_llm)

Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character-level start:end indices of the quote in the source text
  confidence  : LLM confidence score (0=not mentioned/negated, 1=indirect, 2=clear)



,doc_id,construct,quote,speaker_id,quote_index,confidence
0,anxiety_An0nus3r123,anxiety,I know my anxiety has made it hard for me to g...,None,71:138,2
1,anxiety_An0nus3r123,anxiety,Because again i know anxiety can make it hard ...,None,432:513,2
2,anxiety_An0nus3r123,shame,I'm not a interesting person,None,804:832,2
3,anxiety_An0nus3r123,shame,"And i feel like saying watching tv, reading, ...",None,514:613,1
4,anxiety_Anon1234567899754,anxiety,I feel so stressed and uncomfortable in my own...,None,194:258,2
...,...,...,...,...,...,...
243,depression_JovialDemon01,depressed_mood,anxiety/depression medication,None,584:613,2
244,depression_Domihoes,depressed_mood,deep depression,None,51:66,2
245,depression_Domihoes,depressed_mood,lowest ive ever been in my life right now ( re...,None,93:149,2
246,depression_Domihoes,anxiety,anxiety,None,233:240,2


### Retry Failed Documents

If any documents failed — due to API errors, rate limits, or malformed responses — you can retry only the failed ones without re-processing documents that already succeeded. Uncomment the code below and update `output_dir` to point to the timestamped directory from your run above.

In [ ]:

# recovered_results, recovered_meta, remaining_errors = analyzer.retry_errors(
#     output_dir="path/to/my_analysis_1_YYYY-MM-DD_HHMMSS",
#     codebook_path=codebook_path,
# )

## Step 2A: Human Coding

For the comparison step you need a second set of codings to compare against the LLM output. In a real study this would be a directory of human-coded JSONs — one per document, with the same filenames as your input files.

The human coding JSONs must follow the same schema as the LLM coding output: a `document_id` field and an `instances` list where each instance has at minimum a `construct` and `quote` field.

**If you have real human codings:** skip this cell and set `human_encoding_dir` in Step 3 to point directly to your human coding directory.

**If you are testing the pipeline:** run a second LLM coding pass as a stand-in for human data, as shown below. Because the same model is run twice on the same documents, agreement will be high but not perfect — the LLM introduces some variability between runs, which makes this a reasonable functional test of the full pipeline.

In [ ]:
# Example: use a second LLM run as a dummy human coder
human_analyzer = PychometricsAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

data_type = "csv"

if data_type == "csv":
    human_results, human_metadata, human_errors = analyzer.analyze_csv(
        csv_path="reddit_autism_anxiety_depression.csv",
        text_column="post",
        subreddit_column="subreddit",
        author_column="author",
        codebook_path=codebook_path,
        output_dir="human_reddit_coding",
    )
elif data_type =="interview":
     human_results, human_metadata, human_errors = analyzer.analyze_directory(
        input_dir=input_dir,
        codebook_path=codebook_path,
        output_dir="human_coding",  # timestamped suffix added automatically
    )
else: 
    print("Please set data type to either reddit or interview!")

Processing [1/60]: anxiety_An0nus3r123
  ✓ Found 4 construct instances
  Progress: Successful 1/60; Errors 0/60
Processing [2/60]: anxiety_Anon1234567899754
  ✓ Found 2 construct instances
  Progress: Successful 2/60; Errors 0/60
Processing [3/60]: anxiety_vwerfel
  ✓ Found 3 construct instances
  Progress: Successful 3/60; Errors 0/60
Processing [4/60]: anxiety_LostWildOne
  ✓ Found 7 construct instances
  Progress: Successful 4/60; Errors 0/60
Processing [5/60]: anxiety_frauleinfarts
  ✓ Found 2 construct instances
  Progress: Successful 5/60; Errors 0/60
Processing [6/60]: anxiety_neo-mancer
  ✓ Found 2 construct instances
  Progress: Successful 6/60; Errors 0/60
Processing [7/60]: anxiety_4ssCr4ck
  ✓ Found 7 construct instances
  Progress: Successful 7/60; Errors 0/60
Processing [8/60]: anxiety_NeptuneOW
  ✓ Found 3 construct instances
  Progress: Successful 8/60; Errors 0/60
Processing [9/60]: anxiety_zhvng666
  ✓ Found 7 construct instances
  Progress: Successful 9/60; Errors 0/

In [6]:
format_coding_table(human_results)

Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character-level start:end indices of the quote in the source text
  confidence  : LLM confidence score (0=not mentioned/negated, 1=indirect, 2=clear)



,doc_id,construct,quote,speaker_id,quote_index,confidence
0,anxiety_An0nus3r123,anxiety,I know my anxiety has made it hard for me to g...,None,71:138,2
1,anxiety_An0nus3r123,anxiety,Because again i know anxiety can make it hard ...,None,432:513,2
2,anxiety_An0nus3r123,shame,I'm not a interesting person,None,804:832,2
3,anxiety_An0nus3r123,shame,"And i feel like saying watching tv, reading, ...",None,514:613,1
4,anxiety_Anon1234567899754,anxiety,I feel so stressed and uncomfortable in my own...,None,194:258,2
...,...,...,...,...,...,...
259,depression_JovialDemon01,rumination,"I just keep asking myself, what's wrong with m...",None,392:528,2
260,depression_Domihoes,depressed_mood,deep depression,None,51:66,2
261,depression_Domihoes,depressed_mood,lowest ive ever been in my life right now ( re...,None,93:149,2
262,depression_Domihoes,anxiety,anxiety,None,233:240,2


## Step 2B: Load Human Coding from Dedoose

In most real-world use cases, you won’t be comparing two LLM runs — instead, you’ll compare the LLM’s coding against human-coded data.

If you are using Dedoose, you can export your coded excerpts as an `.xlsx` file and load them directly into pychometrics.

Under the hood, this step converts the Dedoose export into the same internal format used by the LLM (`AnalysisResult`). This ensures that human and LLM outputs are directly comparable without any additional formatting.

**Important:** The `Media Title` column in your Dedoose export must match the document IDs used in your LLM analysis (typically the filename without extension). If these don’t match, the comparison will not align correctly.

In this step, we:
- Load the Dedoose export into a pandas DataFrame
- Convert it into pychometrics format
- Store it as `human_results` for comparison

In [ ]:
# Load Dedoose export (xlsx)
dedoose_df = pd.read_excel("path/to/your_dedoose_export.xlsx")

# Convert to pychometrics format
human_results = load_dedoose_dataframe(dedoose_df, "path/to/your_codebook.json")

## Step 3: Comparison

This step aligns the human and LLM codings for each document. For each construct, an LLM matcher reads all quotes from both coders and decides which human quotes and LLM quotes refer to the same passage or idea — even if the wording differs (paraphrase).

Each instance in the resulting DataFrame is classified as one of three statuses:
- **`matched`** — both coders identified this passage (True Positive, TP). Includes a `match_confidence` score from the matcher and a `span_overlap` score (Jaccard overlap of character indices)
- **`human_only`** — the human identified this passage but the LLM missed it (False Negative, FN)
- **`llm_only`** — the LLM identified this passage but the human missed it (False Positive, FP)

**Important:** update `llm_encoding_dir` and `human_encoding_dir` below to the `codings/` subdirectories of your runs from Steps 1 and 2. The paths include timestamps so they will be different each time you run.

You can optionally use a different model for the matching step than for coding — for example a more capable model for matching if accuracy is critical. Set `match_model` accordingly.

In [ ]:
# Update these paths to point to the codings/ subdirectories of your runs
# If this notebook is run multiple times, remember to update these paths.
llm_encoding_dir   = ""
human_encoding_dir = ""
#change the match model if you want different models for encoding and comparison. 
comparator = PychometricsComparer(
    api_key=api_key,
    match_model=llm_model, 
)

comparison_results = comparator.compare_directories(
    human_dir=human_encoding_dir,
    llm_dir=llm_encoding_dir,
    output_dir="comparison_run",  # optional; saves JSON comparison files
)

## Step 4: Build Summary Tables

Now that we have compared the LLM coding to the human coding, we can create summary tables that make the results easier to read.

These tables answer slightly different questions:

- **Concatenated summary:** If we pool all coded examples across all documents together, how well did the LLM perform for each construct?
- **Per-document summary:** How well did the LLM perform within each individual document?
- **Weighted summary:** What does performance look like across documents when documents with more coded material count more heavily?

### Important note about the metrics

This is an open-ended coding task, not a standard classification task with true negatives.

We observe:
- True positives (TP)
- False positives (FP)
- False negatives (FN)

But we do NOT have true negatives (TN), so:
- Specificity is not meaningful
- ROC AUC is not appropriate

Metrics like precision, recall, F1, kappa, and PR AUC are used instead.


### 4.1 Row-Level Comparison DataFrame

The raw DataFrame `df` has one row per coded instance across all documents. Each row represents a single quote that was identified by at least one coder, and includes the full quote text, character indices, and TP/FP/FN indicators. This is the most granular view and is useful for inspecting specific mismatches — for example, filtering to `status == 'llm_only'` to review what the LLM found that the human missed, or `status == 'human_only'` to review misses.

In [10]:
# Row-level view: every matched/human_only/llm_only instance across all interviews
comparison_table = format_comparison_table(comparison_results)
comparison_table

Comparison Table
----------------
One row per coded instance across all interviews. Each row represents a quote
identified by at least one coder, with its classification and match details.

Columns:
  doc_id           : interview identifier
  construct        : psychological construct the instance belongs to
  status           : matched (TP), human_only (FN), or llm_only (FP)
  human_quote      : quote extracted by the human coder
  llm_quote        : quote extracted by the LLM
  human_indices    : character-level start:end indices of the human quote in the source text
  llm_indices      : character-level start:end indices of the LLM quote in the source text
  paraphrase       : True if the matched quotes differ meaningfully in wording
  span_overlap     : Jaccard overlap between human and LLM character spans (matched rows only)
  match_confidence : LLM matcher confidence that the two quotes refer to the same passage
  tp/fp/fn         : binary indicators for this row's contribution to

,doc_id,construct,status,human_quote,llm_quote,human_indices,llm_indices,paraphrase,span_overlap,match_confidence,tp,fp,fn
0,anxiety_4ssCr4ck,anxiety,matched,Is this anxiety?,Is this anxiety?,0:16,0:16,False,1.0,1.0,1,0,0
1,anxiety_4ssCr4ck,anxiety,matched,people that really have anxiety,people that really have anxiety,71:102,71:102,False,1.0,1.0,1,0,0
2,anxiety_4ssCr4ck,anxiety,matched,anxious,anxious,299:306,299:306,False,1.0,1.0,1,0,0
3,anxiety_4ssCr4ck,depressed_mood,matched,crying,crying aswell,372:378,372:385,False,0.461538,1.0,1,0,0
4,anxiety_4ssCr4ck,panic,matched,made me literally panic,made me literally panic,254:277,254:277,False,1.0,1.0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
284,depression_scrappydoosghost,shame,matched,I SHOULD die.,I SHOULD die,425:438,425:437,False,0.923077,1.0,1,0,0
285,depression_scrappydoosghost,shame,matched,"I hate me so much,",I hate me so much,453:471,453:470,False,0.944444,1.0,1,0,0
286,depression_scrappydoosghost,shame,matched,so what do I deserve? Death?,so what do I deserve? Death?,594:622,594:622,False,1.0,1.0,1,0,0
287,depression_scrappydoosghost,shame,llm_only,NaN,I don’t deserve any of this I swear- what did ...,NaN,472:523,None,NaN,NaN,0,1,0


### 4.2 Per-Document Metrics

One row per (document, construct) combination. Shows raw TP/FP/FN counts and all metrics broken down by document. Use this table to identify whether performance issues are consistent across documents or driven by specific documents. Constructs that do not appear in a given document are absent from this table — they do not appear as zero rows. PR AUC is computed per (document, construct) where there are enough positive and negative predictions; otherwise it is `NaN`.

In [11]:
per_interview, concatenated, weighted_summary = compute_summary_tables(comparison_table)

In [12]:
# Per-interview metrics: one row per (interview, construct)
format_per_interview(per_interview)

Per-Interview Metrics
---------------------
One row per (interview, construct) combination. Constructs that did not appear
in a given interview are absent — they do not appear as zero rows.

Columns:
  doc_id       : interview identifier
  construct    : psychological construct
  tp/fp/fn     : true positives, false positives, false negatives for this interview-construct
  union        : total coded instances (TP + FP + FN)
  sensitivity  : TP / (TP + FN)
  precision    : TP / (TP + FP)
  f1           : harmonic mean of sensitivity and precision
  cohens_kappa : agreement beyond chance (TN=0 convention)
  pabak        : prevalence-adjusted kappa
  pr_auc       : area under precision-recall curve; NaN where insufficient label classes



,doc_id,construct,tp,fp,fn,union,sensitivity,precision,f1,cohens_kappa,pabak,pr_auc
0,anxiety_4ssCr4ck,anxiety,3,0,0,3,1.0,1.0,1.00,1.0,1.0,NaN
1,anxiety_4ssCr4ck,depressed_mood,1,0,0,1,1.0,1.0,1.00,1.0,1.0,NaN
2,anxiety_4ssCr4ck,panic,2,0,0,2,1.0,1.0,1.00,1.0,1.0,NaN
3,anxiety_4ssCr4ck,rumination,0,1,0,1,NaN,0.0,0.00,0.0,-1.0,NaN
4,anxiety_4ssCr4ck,shame,1,0,0,1,1.0,1.0,1.00,1.0,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
133,depression_nefariousbeliever,depressed_mood,1,0,0,1,1.0,1.0,1.00,1.0,1.0,NaN
134,depression_nefariousbeliever,shame,2,0,0,2,1.0,1.0,1.00,1.0,1.0,NaN
135,depression_scrappydoosghost,anhedonia,1,0,0,1,1.0,1.0,1.00,1.0,1.0,NaN
136,depression_scrappydoosghost,shame,4,1,0,5,1.0,0.8,0.89,0.0,0.6,1.0


### 4.3 Concatenated Metrics

One row per construct, with counts pooled across all documents before metrics are computed. Think of this as treating the entire dataset as a single document. This table answers: *overall, across all documents, how did the LLM perform on each construct?* The `number of docum [p5–p95]` column shows how many documents contained at least one instance of the construct, along with the 5th and 95th percentile of instance counts — giving a sense of how common and variable the construct is across your data.

In [13]:
# Concatenated metrics: counts pooled across all interviews, one row per construct
format_concatenated(concatenated)

Concatenated Metrics
--------------------
One row per construct. TP/FP/FN counts are pooled across all interviews before
metrics are computed — treats the entire dataset as a single document.

Columns:
  tp/fp/fn                          : total counts pooled across all interviews
  sensitivity                       : TP / (TP + FN) — proportion of human-coded instances the LLM found
  precision                         : TP / (TP + FP) — proportion of LLM-coded instances that were correct
  f1                                : harmonic mean of sensitivity and precision
  cohens_kappa                      : agreement beyond chance; computed over coded instances only (TN=0 convention)
  pabak                             : prevalence-adjusted kappa; more stable when construct prevalence is low
  pr_auc                            : area under precision-recall curve; ranks LLM predictions by match confidence
  interviews_with_construct [p5-p95]: number of interviews containing the construct,

,construct,tp,fp,fn,sensitivity,precision,f1,cohens_kappa,pabak,pr_auc,interviews_with_construct [p5-p95]
0,agitation,6,0,2,0.75,1.00,0.86,0.00,0.50,—,2 [0.00-0.00]
1,anhedonia,22,6,3,0.88,0.79,0.83,-0.15,0.42,1.00,11 [0.00-3.00]
2,anxiety,64,8,4,0.94,0.89,0.91,-0.08,0.68,1.00,33 [0.00-4.00]
3,depressed_mood,33,5,10,0.77,0.87,0.81,-0.16,0.38,1.00,23 [0.00-3.40]
4,guilt,12,0,1,0.92,1.00,0.96,0.00,0.85,—,9 [0.00-1.00]
5,panic,16,1,3,0.84,0.94,0.89,-0.08,0.60,1.00,14 [0.00-2.00]
6,rumination,23,1,7,0.77,0.96,0.85,-0.06,0.48,1.00,17 [0.00-3.00]
7,shame,45,4,11,0.80,0.92,0.86,-0.11,0.50,1.00,27 [0.00-4.40]
8,trauma_and_ptsd,2,0,0,1.00,1.00,1.00,1.00,1.00,—,2 [0.00-0.00]
9,Overall,223,25,41,0.84,0.90,0.87,-0.12,0.54,1.00,53 [0.00-3.00]


### 4.4 Weighted Summary

One row per construct. Metrics are first computed per document, then summarized as weighted median [min–max] across documents, where each document is weighted by its union size (TP + FP + FN) — so documents with more coded instances contribute more to the median. This table answers: *what does performance typically look like at the individual document level?* The [min–max] brackets show the range across documents, which is useful for spotting constructs with highly variable performance. A `—` in the PR AUC column means there were insufficient label classes in every document to compute the metric for that construct.

In [14]:
# Weighted summary: median [min-max] weighted by union size, with n_docs prevalence column
format_weighted_summary(weighted_summary)

Weighted Summary
----------------
One row per construct. Metrics are computed per interview first, then summarized
as weighted median [min–max] across interviews, weighted by union size (TP+FP+FN).

Columns:
  tp/fp/fn                          : total true positives, false positives, false negatives across all interviews
  sensitivity                       : TP / (TP + FN) — proportion of human-coded instances the LLM found
  precision                         : TP / (TP + FP) — proportion of LLM-coded instances that were correct
  f1                                : harmonic mean of sensitivity and precision
  cohens_kappa                      : agreement beyond chance; computed over coded instances only (TN=0 convention)
  pabak                             : prevalence-adjusted kappa; more stable when construct prevalence is low
  pr_auc                            : area under precision-recall curve; ranks LLM predictions by match confidence
  interviews_with_construct [p5-p95]: numbe

,construct,tp,fp,fn,sensitivity,precision,f1,cohens_kappa,pabak,pr_auc,interviews_with_construct [p5–p95]
0,anxiety,64,8,4,1.00 [0.50–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [-0.50–1.00],1.00 [-1.00–1.00],1.00 [1.00–1.00],33 [0.00–4.00]
1,depressed_mood,33,5,10,1.00 [0.00–1.00],1.00 [0.00–1.00],0.89 [0.00–1.00],0.00 [-1.00–1.00],0.60 [-1.00–1.00],1.00 [1.00–1.00],23 [0.00–3.40]
2,panic,16,1,3,1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [-1.00–1.00],—,14 [0.00–2.00]
3,rumination,23,1,7,1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [-1.00–1.00],—,17 [0.00–3.00]
4,shame,45,4,11,1.00 [0.00–1.00],1.00 [0.60–1.00],0.86 [0.00–1.00],0.00 [-0.29–1.00],0.50 [-1.00–1.00],1.00 [1.00–1.00],27 [0.00–4.40]
5,trauma_and_ptsd,2,0,0,1.00 [1.00–1.00],1.00 [1.00–1.00],1.00 [1.00–1.00],1.00 [1.00–1.00],1.00 [1.00–1.00],—,2 [0.00–0.00]
6,anhedonia,22,6,3,0.88 [0.50–1.00],0.88 [0.00–1.00],0.88 [0.00–1.00],-0.12 [-0.50–1.00],0.56 [-1.00–1.00],1.00 [1.00–1.00],11 [0.00–3.00]
7,guilt,12,0,1,1.00 [0.00–1.00],1.00 [1.00–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [-1.00–1.00],—,9 [0.00–1.00]
8,agitation,6,0,2,0.71 [0.71–1.00],1.00 [1.00–1.00],0.83 [0.83–1.00],0.00 [0.00–1.00],0.43 [0.43–1.00],—,2 [0.00–0.00]
9,Overall,223,25,41,1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [-1.00–1.00],1.00 [-1.00–1.00],1.00 [1.00–1.00],53 [0.00–3.00]
